In [1]:
!pip install open3d
!pip install plyfile

  Using cached plyfile-1.1.4-py3-none-any.whl.metadata (43 kB)
Using cached plyfile-1.1.4-py3-none-any.whl (36 kB)


In [2]:
import os
import glob
import numpy as np
import open3d as o3d
from plyfile import PlyData
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from scipy.spatial import KDTree

# ==============================================================================
# 1-2. ЗАГРУЗКА И ВЫСОКОТОЧНАЯ ПРЕДОБРАБОТКА (Размер вокселя уменьшен для точности)
# ==============================================================================

def load_and_preprocess_pcd(file_path, min_points=10, voxel_size=0.02):
    """
    Загрузка PLY, удаление шума, надежный numpy-downsampling и расчет нормалей.
    """
    if not os.path.exists(file_path):
        return None, None

    try:
        plydata = PlyData.read(file_path)
        vertex_data = plydata['vertex']
        points = np.vstack([vertex_data['x'], vertex_data['y'], vertex_data['z']]).T

        available_properties = [p.name for p in vertex_data.properties]
        labels = None
        for name in ["scalar_Label", "label", "class", "classes", "segment"]:
            if name in available_properties:
                labels = np.array(vertex_data[name]).flatten().astype(int)
                break

        if labels is None:
            return None, None

    except Exception:
        return None, None

    if len(points) < min_points:
        return None, None

    # ---- 1. Статистическое удаление шумов через Open3D ----
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=25, std_ratio=2.0)
    ind = np.asarray(ind)
    points = points[ind]
    labels = labels[ind]

    # ---- 2. DOWNSAMPLING НА NUMPY ----
    voxel_coords = np.floor(points / voxel_size).astype(int)

    _, down_indices = np.unique(voxel_coords, axis=0, return_index=True)

    points_final = points[down_indices]
    labels_final = labels[down_indices]

    # ---- 3. Создание финального геометрического объекта и расчет нормалей ----
    pcd_final = o3d.geometry.PointCloud()
    pcd_final.points = o3d.utility.Vector3dVector(points_final)

    pcd_final.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    pcd_final.orient_normals_consistent_tangent_plane(k=15)

    return pcd_final, labels_final


# ==============================================================================
# 3. ФОРМИРОВАНИЕ СЕГМЕНТОВ
# ==============================================================================

def split_into_segments(pcd, labels, min_segment_points=40):
    segments = {}
    unique_labels = np.unique(labels)
    points_np = np.asarray(pcd.points)
    normals_np = np.asarray(pcd.normals)

    for label in unique_labels:
        idx = np.where(labels == label)[0]
        if len(idx) < min_segment_points:
            continue

        seg_pcd = o3d.geometry.PointCloud()
        seg_pcd.points = o3d.utility.Vector3dVector(points_np[idx])
        seg_pcd.normals = o3d.utility.Vector3dVector(normals_np[idx])
        segments[label] = seg_pcd

    return segments

# ==============================================================================
# 4-5. ГЕОМЕТРИЧЕСКИЙ АНАЛИЗ И КЛАССИФИКАЦИЯ (Включая все 4 типа из ТЗ)
# ==============================================================================

def analyze_and_classify_segment(seg_pcd):
    pts = np.asarray(seg_pcd.points)
    if len(pts) < 5:
        return "сложная геометрия"

    covariance_matrix = np.cov(pts.T)
    eigenvalues, _ = np.linalg.eigh(covariance_matrix)
    eigenvalues = np.sort(eigenvalues)[::-1]

    lam1, lam2, lam3 = eigenvalues
    sum_lam = np.sum(eigenvalues) + 1e-8

    linearity = (lam1 - lam2) / sum_lam
    planarity = (lam2 - lam3) / sum_lam
    sphericity = lam3 / sum_lam

    normals = np.asarray(seg_pcd.normals)
    avg_normal = np.mean(normals, axis=0)
    normal_variance = np.mean(1.0 - np.dot(normals, avg_normal)**2)

    if planarity > 0.65 and normal_variance < 0.15:
        return "плоская поверхность"
    elif linearity > 0.5:
        return "трубчатый объект"
    elif sphericity > 0.3:
        return "сферическая форма"
    else:
        return "сложная геометрия"

# ==============================================================================
# 6-7. АДАПТИВНАЯ РЕКОНСТРУКЦИЯ (Возвращены Poisson, BP и Alpha Shapes)
# ==============================================================================

def reconstruct_segment(seg_pcd, geom_type):
    """
    Выбор алгоритма на основе геометриию
    """
    try:
        distances = seg_pcd.compute_nearest_neighbor_distance()
        avg_dist = np.mean(distances) if len(distances) > 0 else 0.02

        if geom_type == "плоская поверхность":
            # 1. Ball Pivoting для плоскостей (сохраняет четкие границы)
            radii = [avg_dist, avg_dist * 2.0]
            mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
                seg_pcd, o3d.utility.DoubleVector(radii)
            )
        elif geom_type in ["трубчатый объект", "сферическая форма"]:
            # 2. Poisson Surface Reconstruction для гладких и замкнутых тел
            mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
                seg_pcd, depth=8, linear_fit=True
            )
            vertices_to_remove = densities < np.quantile(densities, 0.08)
            mesh.remove_vertices_by_mask(vertices_to_remove)
        else:
            # 3. Alpha Shapes для вогнутых и сложных промышленных узлов
            alpha = avg_dist * 4.0
            mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(seg_pcd, alpha)

        mesh.remove_degenerate_triangles()
        mesh.remove_duplicated_triangles()
        return mesh
    except Exception:
        return None

# ==============================================================================
# 9. ОЦЕНКА КАЧЕСТВА (Метрика Chamfer Distance / Погрешность RMSE)
# ==============================================================================

def evaluate_mesh_quality(source_pcd, reconstructed_mesh):
    if reconstructed_mesh is None or len(reconstructed_mesh.triangles) == 0:
        return None
    sampled_pcd = reconstructed_mesh.sample_points_uniformly(number_of_points=len(source_pcd.points))
    dist_source_to_mesh = source_pcd.compute_point_cloud_distance(sampled_pcd)
    rmse = np.sqrt(np.mean(np.square(dist_source_to_mesh)))
    return rmse

# ==============================================================================
# 8. СКВОЗНОЙ КОНВЕЙЕР И СБОРКА МОДЕЛИ
# ==============================================================================

def process_single_cloud(file_path):
    pcd, labels = load_and_preprocess_pcd(file_path)
    if pcd is None: return None, None, None

    segments = split_into_segments(pcd, labels)
    combined_mesh = o3d.geometry.TriangleMesh()
    file_rmse_list = []

    for label, seg_pcd in segments.items():
        geom_type = analyze_and_classify_segment(seg_pcd)
        mesh = reconstruct_segment(seg_pcd, geom_type)

        if mesh is not None and len(mesh.triangles) > 0:
            rmse = evaluate_mesh_quality(seg_pcd, mesh)
            if rmse is not None:
                file_rmse_list.append(rmse)

            combined_mesh += mesh

    combined_mesh.compute_vertex_normals()

    mean_file_rmse = np.mean(file_rmse_list) if file_rmse_list else 0.0
    return combined_mesh, mean_file_rmse


def process_and_save(file_path, folder_name, output_dir):
    try:
        final_model, rmse = process_single_cloud(file_path)
        if final_model is not None and len(final_model.triangles) > 0:
            file_base = os.path.splitext(os.path.basename(file_path))[0]
            save_path = os.path.join(output_dir, f"{file_base}_mesh.ply")
            o3d.io.write_triangle_mesh(save_path, final_model)
            return rmse
    except Exception:
        pass
    return None

In [3]:
# ==============================================================================
# ОСНОВНОЙ БЛОК: ЗАПУСК ДЛЯ ОДНОЙ КОНКРЕТНОЙ ПОДПАПКИ
# ==============================================================================
if __name__ == "__main__":
    DATASET_DIR = "/content/drive/MyDrive/MIPT_DS2"

    TARGET_FOLDER_NAME = "15"

    OUTPUT_DIR = f"/content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/reconstructed_{TARGET_FOLDER_NAME}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    target_path = "/content/drive/MyDrive/MIPT_DS2/15"

    pcd_files = sorted(glob.glob(os.path.join(target_path, "*.ply")))
    total_files = len(pcd_files)

    print(f"Выбрана папка: '{TARGET_FOLDER_NAME}'. Найдено файлов: {total_files}")
    print("Запуск прецизионного конвейера обработки...")

    all_rmse_values = []

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {
            executor.submit(process_and_save, f, TARGET_FOLDER_NAME, OUTPUT_DIR): f
            for f in pcd_files
        }

        with tqdm(total=total_files, desc=f"Обработка {TARGET_FOLDER_NAME}") as pbar:
            for future in as_completed(futures):
                file_rmse = future.result()
                if file_rmse is not None:
                    all_rmse_values.append(file_rmse)
                pbar.update(1)
    if all_rmse_values:
        print(f"\n" + "="*50)
        print(f"СТАТИСТИКА ПО ПАПКЕ '{TARGET_FOLDER_NAME}':")
        print(f"Успешно обработано объектов: {len(all_rmse_values)} из {total_files}")
        print(f"Средняя ошибка реконструкции (Mean RMSE): {np.mean(all_rmse_values):.5f} м.")
        print(f"Минимальная ошибка (Best Mesh): {np.min(all_rmse_values):.5f} м.")
        print(f"Максимальная ошибка (Worst Mesh): {np.max(all_rmse_values):.5f} м.")
        print("="*50)
    else:
        print("\nОшибка: Ни один файл не был успешно реконструирован. Проверьте структуру папок.")

Выбрана папка: '15'. Найдено файлов: 500
Запуск прецизионного конвейера обработки...


Обработка 15: 100%|██████████| 500/500 [21:59<00:00,  2.64s/it]


СТАТИСТИКА ДЛЯ ОТЧЕТА ПО ПАПКЕ '15':
Успешно обработано объектов: 500 из 500
Средняя ошибка реконструкции (Mean RMSE): 5.97629 м.
Минимальная ошибка (Best Mesh): 4.33045 м.
Максимальная ошибка (Worst Mesh): 11.09924 м.


In [ ]:
import os
import glob
import numpy as np
import open3d as o3d
from plyfile import PlyData
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from scipy.spatial import KDTree

if __name__ == "__main__":
    DATASET_DIR = "/content/drive/MyDrive/MIPT_DS2"
    TARGET_FOLDER_NAME = "15"

    OUTPUT_DIR = f"/content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/reconstructed_{TARGET_FOLDER_NAME}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    pcd_files = sorted(glob.glob(os.path.join(DATASET_DIR, TARGET_FOLDER_NAME, "*.ply")))

    all_rmse = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(process_and_save, f, OUTPUT_DIR): f for f in pcd_files}
        with tqdm(total=len(pcd_files), desc=f"Расчет {TARGET_FOLDER_NAME}") as pbar:
            for future in as_completed(futures):
                res = future.result()
                if res is not None: all_rmse.append(res)
                pbar.update(1)

    print(f"\n==================================================")
    print(f"СТАТИСТИЧЕСКИЙ АНАЛИЗ:")
    print(f"Целевая подпапка: {TARGET_FOLDER_NAME}")
    print(f"Успешность построения конвейера: {len(all_rmse)} / {len(pcd_files)} файлов")
    print(f"Средняя погрешность Chamfer Distance (Mean RMSE): {np.mean(all_rmse):.5f} м.")
    print(f"Минимальная зафиксированная погрешность (Best): {np.min(all_rmse):.5f} м.")
    print(f"Максимальная зафиксированная погрешность (Worst): {np.max(all_rmse):.5f} м.")
    print(f"==================================================")


Расчет 15: 100%|██████████| 500/500 [24:12<00:00,  2.90s/it]


СТАТИСТИЧЕСКИЙ АНАЛИЗ ДЛЯ РАЗДЕЛА МЕТРИК КАЧЕСТВА:
Целевая подпапка: 15
Успешность построения конвейера: 500 / 500 файлов
Средняя погрешность Chamfer Distance (Mean RMSE): 5.96799 м.
Минимальная зафиксированная погрешность (Best): 4.38352 м.
Максимальная зафиксированная погрешность (Worst): 9.80804 м.


In [ ]:
import plotly.graph_objects as go

sample_file = pcd_files[0]
mesh, rmse, source_pcd = process_single_cloud(sample_file)

if mesh is not None:
    verts = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)

    fig = go.Figure(data=[
        go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=triangles[:, 0], j=triangles[:, 1], k=triangles[:, 2],
            opacity=0.8,
            color='lightblue',
            flatshading=True,
            name="Реконструированный Меш"
        )
    ])

    fig.update_layout(
        title=f"Интерактивная модель: {os.path.basename(sample_file)} (RMSE: {rmse:.4f} м.)",
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()


In [ ]:
!zip -r reconstructed_models.zip /content/drive/MyDrive/mipt_tsitis_cv_26/lab2_results/reconstructed_valve

from google.colab import files
files.download('reconstructed_models.zip')
